In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
import seaborn as sns

df = pd.read_csv("Task 3 and 4_Loan_Data.csv")

In [15]:
# 2. Isolate the relevant columns
# We only need the FICO score and whether they defaulted (0 or 1)
data = df[['fico_score', 'default']].copy()

# 3. Sort the data by FICO score in ascending order
# This is crucial so we can slice the data into contiguous intervals later
data = data.sort_values(by='fico_score').reset_index(drop=True)

# Display the first few rows to verify it worked
print(data.head())

   fico_score  default
0         408        0
1         409        1
2         418        1
3         425        1
4         438        1


## Decision Tree Binning

In [18]:
# 1. Define how many buckets/ratings Charlie wants
NUM_BUCKETS = 3

# 2. Set up our features (X) and target (y)
# scikit-learn expects X to be a 2D array/dataframe, so we keep the double brackets
X = data[['fico_score']] 
y = data['default']

# 3. Create and train the Decision Tree
# max_leaf_nodes tells the tree exactly how many final buckets to make
tree_model = DecisionTreeClassifier(max_leaf_nodes=NUM_BUCKETS, random_state=42)
tree_model.fit(X, y)

# 4. Extract the optimal boundaries (splits) chosen by the tree
# The tree stores split thresholds. We filter out the leaf nodes (represented by -2)
thresholds = tree_model.tree_.threshold[tree_model.tree_.threshold != -2]

# Sort the boundaries from lowest to highest FICO score
optimal_boundaries = np.sort(thresholds)

print("Optimal FICO Score Boundaries found by the tree:")
print(optimal_boundaries)

Optimal FICO Score Boundaries found by the tree:
[580.5 640.5]


## Rating Map

In [20]:
# 1. Create a function to map a single FICO score to a rating
def get_rating(fico_score):
    # Rule: Lower rating = Better credit score
    if fico_score > 640.5:
        return 1  # Excellent FICO -> Rating 1
    elif fico_score > 580.5:
        return 2  # Fair FICO -> Rating 2
    else:
        return 3  # Poor FICO -> Rating 3

# 2. Apply this function to create a new 'rating' column in our dataset
data['rating'] = data['fico_score'].apply(get_rating)

# 3. Prove that it works by calculating the Probability of Default (PD) for each rating
rating_summary = data.groupby('rating').agg(
    total_borrowers=('default', 'count'),
    defaults=('default', 'sum'),
    probability_of_default=('default', 'mean') # mean of 0s and 1s gives the exact probability
).reset_index()

# Formatting the probability as a percentage for readability
rating_summary['probability_of_default'] = (rating_summary['probability_of_default'] * 100).round(2).astype(str) + '%'

print("Final Rating Map Summary:")
print(rating_summary)

Final Rating Map Summary:
   rating  total_borrowers  defaults probability_of_default
0       1             4854       413                  8.51%
1       2             3438       703                 20.45%
2       3             1708       735                 43.03%
